In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import importlib
from config import *
from simulation.aggregate_metrics import aggregate_metrics


In [2]:
import pandas as pd
from pathlib import Path

# Folder containing the Excel files
folder = Path("../results")

# Find all Excel files starting with "results_"
files = sorted(folder.glob("results_*.xlsx"))

# Read and concatenate
metrics = pd.concat(
    (pd.read_excel(file, sheet_name="metrics") for file in files),
    ignore_index=True
)

print(f"Loaded {len(files)} files.")
print(metrics.head())

Loaded 108 files.
  method hv_selection       selection_type   bound_estimator  sample_size  \
0    MUS      nothing  systematic_sampling  Poisson_Stringer           30   
1    MUS      nothing  systematic_sampling  Poisson_Stringer           30   
2    MUS      nothing  systematic_sampling  Poisson_Stringer           30   
3    MUS      nothing  systematic_sampling  Poisson_Stringer           65   
4    MUS      nothing  systematic_sampling  Poisson_Stringer           65   

   confidence_level   z_score                        Population ID  \
0              0.80  0.841621  BV_15pct_above_SI_F0.05_C0.1_R0.002   
1              0.90  1.281552  BV_15pct_above_SI_F0.05_C0.1_R0.002   
2              0.95  1.644854  BV_15pct_above_SI_F0.05_C0.1_R0.002   
3              0.80  0.841621  BV_15pct_above_SI_F0.05_C0.1_R0.002   
4              0.90  1.281552  BV_15pct_above_SI_F0.05_C0.1_R0.002   

   Population Book Value  Population Error Amount  ...   Real n   Needed n  \
0           3.425979

In [3]:
import re

# Population ID format (see main.py's _population_id()): BVBV_<N>pct_above_SI_F<f>_C<c>_R<r>
# e.g. "BVBV_5pct_above_SI_F0.2_C0.1_R0.01" -> BV_pop="5pct", f_target=0.2, corr_target=0.1, r_target=0.01
pattern = r"^BV_(?P<bv>\d+pct)_above_SI_F(?P<f>[\d.]+)_C(?P<c>[\d.]+)_R(?P<r>[\d.]+)$"
extracted = metrics["Population ID"].astype(str).str.extract(pattern)

metrics["BV_pop"] = extracted["bv"]
metrics["f_target"] = extracted["f"].astype(float)
metrics["corr_target"] = extracted["c"].astype(float)
metrics["r_target"] = extracted["r"].astype(float)

In [4]:
path = RESULTS_DIR / "main_simulation_results.xlsx"
with pd.ExcelWriter(path, engine="xlsxwriter") as writer:
        metrics.to_excel(writer, sheet_name="metrics", index=False)
        for group_col, table in aggregate_metrics(metrics, "main").items():
            table.to_excel(writer, sheet_name=f"agg_{group_col}"[:31], index=True)

### HH special-case rule: coverage and acceptance rate, split by whether it was applied

For each real `results_*.xlsx` file (HH rows only), split iterations into "rule applied" (`ULE_HH != ULE_pred`) and "rule NOT applied", and compute Coverage / Rate of Acceptance separately within each subset. Uses the already-exported results -- no re-run.

In [4]:
import re
import os
from tqdm import tqdm

TE_PERC = 0.02  # matches config.SIMULATION_SETTINGS["TE_perc"]

rule_split_data = pd.DataFrame()
pattern = r"^results_BV_(?P<bv>\d+pct)_above_SI_F(?P<f>[\d.]+)_C(?P<c>[\d.]+)_R(?P<r>[\d.]+)\.xlsx$"

for file in tqdm(files):
    results = pd.read_excel(file, sheet_name="results (€)")
    file_metrics = pd.read_excel(file, sheet_name="metrics")
    m = re.match(pattern, os.path.basename(file))
    if m is None:
        raise ValueError(f"Filename didn't match pattern: {file}")
    extracted = m.groupdict()

    # Ground truth needed for Coverage / Rate of Acceptance -- constant per file,
    # so just take the first row (same source values run_sim.py itself uses:
    # EE_true = population["E"].sum(), TE = TE_perc * population["BV"].sum()).
    EE_true = file_metrics["Population Error Amount"].iloc[0]
    BV_true = file_metrics["Population Book Value"].iloc[0]
    TE = TE_PERC * BV_true

    hh = results[results["bound_estimator"] == "HH"].copy()
    hh["rule_applied"] = hh["ULE_HH"] != hh["ULE_pred"]

    rows = []
    for (ss, cl), grp in hh.groupby(["sample_size", "confidence_level"]):
        applied = grp[grp["rule_applied"]]
        not_applied = grp[~grp["rule_applied"]]
        rows.append({
            "bound_estimator": "HH",
            "sample_size": ss,
            "confidence_level": cl,
            "Coverage (rule applied)": (applied["ULE_pred"] >= EE_true).mean() if len(applied) else float("nan"),
            "Coverage (rule NOT applied)": (not_applied["ULE_pred"] >= EE_true).mean() if len(not_applied) else float("nan"),
            "Rate of Acceptance (rule applied)": (applied["ULE_pred"] <= TE).mean() if len(applied) else float("nan"),
            "Rate of Acceptance (rule NOT applied)": (not_applied["ULE_pred"] <= TE).mean() if len(not_applied) else float("nan"),
            "n (rule applied)": len(applied),
            "n (rule NOT applied)": len(not_applied),
        })
    iteration = pd.DataFrame(rows)
    iteration["BV_pop"] = extracted["bv"]
    iteration["f_target"] = float(extracted["f"])
    iteration["corr_target"] = float(extracted["c"])
    iteration["r_target"] = float(extracted["r"])

    rule_split_data = pd.concat([rule_split_data, iteration], ignore_index=True)

100%|██████████| 108/108 [1:34:49<00:00, 52.68s/it]


In [5]:
rule_split_data

,bound_estimator,sample_size,confidence_level,Coverage (rule applied),Coverage (rule NOT applied),Rate of Acceptance (rule applied),Rate of Acceptance (rule NOT applied),n (rule applied),n (rule NOT applied),BV_pop,f_target,corr_target,r_target
0,HH,30,0.80,1.0,0.064769,0.0,0.935231,1323,8677,15pct,0.05,0.1,0.002
1,HH,30,0.90,1.0,0.064357,0.0,0.935643,1283,8717,15pct,0.05,0.1,0.002
2,HH,30,0.95,1.0,0.070396,0.0,0.929604,1349,8651,15pct,0.05,0.1,0.002
3,HH,65,0.80,1.0,1.000000,0.0,0.000000,8800,1200,15pct,0.05,0.1,0.002
4,HH,65,0.90,1.0,1.000000,0.0,0.000000,8725,1275,15pct,0.05,0.1,0.002
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1615,HH,150,0.90,NaN,0.834400,NaN,0.000000,0,10000,5pct,0.50,0.5,0.040
1616,HH,150,0.95,NaN,0.886100,NaN,0.000000,0,10000,5pct,0.50,0.5,0.040
1617,HH,200,0.80,NaN,0.744100,NaN,0.000000,0,10000,5pct,0.50,0.5,0.040
1618,HH,200,0.90,NaN,0.837800,NaN,0.000000,0,10000,5pct,0.50,0.5,0.040


In [6]:
# Merge into the same "metrics" table main_simulation_results.xlsx is built from, so the
# output is "main_simulation_results.xlsx plus these columns" rather than a separate document.
metrics_with_rule_split = metrics.merge(
    rule_split_data,
    on=["BV_pop", "f_target", "corr_target", "r_target", "sample_size", "confidence_level", "bound_estimator"],
    how="left",
)

path = RESULTS_DIR / "main_simulation_results_with_rule_split.xlsx"
with pd.ExcelWriter(path, engine="xlsxwriter") as writer:
    metrics_with_rule_split.to_excel(writer, sheet_name="metrics", index=False)
    for group_col, table in aggregate_metrics(metrics_with_rule_split, "main").items():
        table.to_excel(writer, sheet_name=f"agg_{group_col}"[:31], index=True)

print(f"Written -> {path}")
metrics_with_rule_split

Written -> C:\Users\isabe\Desktop\PhD\Articles\A1. Artigo MUS standard\MUS_Article\results\main_simulation_results_with_rule_split.xlsx


,method,hv_selection,selection_type,bound_estimator,sample_size,confidence_level,z_score,Population ID,Population Book Value,Population Error Amount,...,n (rule NOT applied),Correct Acceptance,Incorrect Rejection,Incorrect Acceptance,Correct Rejection,Relative Bias of Error Estimation,Relative Precision of Error Estimation,Precision of Error Estimation in %,Relative Bias of Precision Estimation,Relative Precision of Precision Estimation
0,MUS,nothing,systematic_sampling,Poisson_Stringer,30,0.80,0.841621,BV_15pct_above_SI_F0.05_C0.1_R0.002,3.425979e+09,6.851959e+06,...,NaN,0.0,0.0566,NaN,NaN,-0.019236,3.417947,0.006707,0.876709,0.046928
1,MUS,nothing,systematic_sampling,Poisson_Stringer,30,0.90,1.281552,BV_15pct_above_SI_F0.05_C0.1_R0.002,3.425979e+09,6.851959e+06,...,NaN,0.0,0.0600,NaN,NaN,0.045736,5.069877,0.010626,0.863726,0.078815
2,MUS,nothing,systematic_sampling,Poisson_Stringer,30,0.95,1.644854,BV_15pct_above_SI_F0.05_C0.1_R0.002,3.425979e+09,6.851959e+06,...,NaN,0.0,0.0570,NaN,NaN,-0.012301,6.654936,0.013148,0.870240,0.096005
3,MUS,nothing,systematic_sampling,Poisson_Stringer,65,0.80,0.841621,BV_15pct_above_SI_F0.05_C0.1_R0.002,3.425979e+09,6.851959e+06,...,NaN,0.0,0.0068,NaN,NaN,-0.026019,2.334209,0.004550,0.821479,0.066829
4,MUS,nothing,systematic_sampling,Poisson_Stringer,65,0.90,1.281552,BV_15pct_above_SI_F0.05_C0.1_R0.002,3.425979e+09,6.851959e+06,...,NaN,0.0,0.0077,NaN,NaN,0.055255,3.406114,0.007211,0.803143,0.112183
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6475,MUS,iterative,systematic_sampling,HH,150,0.90,1.281552,BV_5pct_above_SI_F0.5_C0.5_R0.04,3.425979e+09,1.370392e+08,...,10000.0,NaN,NaN,0.0,1.0,0.000005,0.201141,0.008046,-0.028058,0.373354
6476,MUS,iterative,systematic_sampling,HH,150,0.95,1.644854,BV_5pct_above_SI_F0.5_C0.5_R0.04,3.425979e+09,1.370392e+08,...,10000.0,NaN,NaN,0.0,1.0,0.000380,0.261459,0.010462,-0.034472,0.484238
6477,MUS,iterative,systematic_sampling,HH,200,0.80,0.841621,BV_5pct_above_SI_F0.5_C0.5_R0.04,3.425979e+09,1.370392e+08,...,10000.0,NaN,NaN,0.0,1.0,0.000115,0.115368,0.004615,-0.024217,0.216886
6478,MUS,iterative,systematic_sampling,HH,200,0.90,1.281552,BV_5pct_above_SI_F0.5_C0.5_R0.04,3.425979e+09,1.370392e+08,...,10000.0,NaN,NaN,0.0,1.0,-0.000261,0.173479,0.006937,-0.012320,0.328037
